Zoe Buck

I used CodeX to create all the code in this notebook via separate prompts. I combined all of the resulting scripts in this notebook, and included each prompt before each section of code. 

# FashionMNIST Classifier

This notebook loads FashionMNIST, creates a reproducible training/validation split, trains a small classifier, and inspects predictions. I used CodeX to create all the code via separate prompts. I included each prompt before each section of code. 

## 1. Imports and device setup

**Prompt:** Use PyTorch and torchvision to prepare the tools needed for loading FashionMNIST, building a classifier, and training it.

In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


## 2. Load FashionMNIST and split it

**Prompt:** Load the full FashionMNIST training dataset and split it into 55,000 training examples and 5,000 validation examples. Use a fixed seed so the split is reproducible.

In [2]:
full_train_dataset = FashionMNIST(
    root="./data", train=True, download=True, transform=ToTensor()
)
train_dataset, validation_dataset = random_split(
    full_train_dataset, [55_000, 5_000],
    generator=torch.Generator().manual_seed(42),
)
print(f"Training examples:   {len(train_dataset)}")
print(f"Validation examples: {len(validation_dataset)}")


Training examples:   55000
Validation examples: 5000


## 3. Inspect an image and its label

**Prompt:** Show the shape and data type of the `x` image sample from the training data, and show the text class label associated with its `y` label index.

In [3]:
x, y = train_dataset[0]
print(f"x shape: {tuple(x.shape)}")
print(f"x data type: {x.dtype}")
print(f"y label index: {y}")
print(f"y text label: {full_train_dataset.classes[y]}")


x shape: (1, 28, 28)
x data type: torch.float32
y label index: 9
y text label: Ankle boot


## 4. Define the classifier and cross-entropy loss

**Prompt:** Create a simple image classifier for 28×28 grayscale images with 10 output classes, then define the cross-entropy loss as `xentropy`.

In [4]:
class FashionMNISTClassifier(nn.Module):
    """A small fully connected classifier for 28x28 images."""
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(), nn.Linear(28 * 28, 256), nn.ReLU(), nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.layers(x)

model = FashionMNISTClassifier().to(device)
xentropy = nn.CrossEntropyLoss()
print(model)
print(f"Loss: {xentropy}")


FashionMNISTClassifier(
  (layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)
Loss: CrossEntropyLoss()


## 5. Create an optimizer and accuracy metric

**Prompt:** Create an optimizer for the model and an accuracy metric with reset, update, and compute methods.

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

class Accuracy:
    def reset(self):
        self.correct = 0
        self.total = 0
    def update(self, logits, targets):
        self.correct += (logits.argmax(dim=1) == targets).sum().item()
        self.total += targets.numel()
    def compute(self):
        return torch.tensor(self.correct / self.total if self.total else 0.0)

metric = Accuracy()
print(f"Optimizer: {optimizer.__class__.__name__}")


Optimizer: Adam


## 6. Define evaluation and training functions

**Prompt:** Implement evaluation with `model.eval()` and disabled gradient tracking. Implement the `train2` workflow to optimize the model and report training loss, training accuracy, and validation accuracy for each epoch.

In [6]:
def evaluate_tm(model, data_loader, metric, device):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            metric.update(model(x_batch), y_batch)
    return metric.compute()


def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
           n_epochs, device):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        metric.reset()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            metric.update(logits, y_batch)
        mean_loss = total_loss / len(train_loader)
        train_accuracy = metric.compute().item()
        valid_accuracy = evaluate_tm(model, valid_loader, metric, device).item()
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(train_accuracy)
        history["valid_metrics"].append(valid_accuracy)
        print(f"Epoch {epoch + 1}/{n_epochs}, train loss: {mean_loss:.4f}, "
              f"train accuracy: {train_accuracy:.4f}, "
              f"validation accuracy: {valid_accuracy:.4f}")
    return history


## 7. Train the model and calculate parameter count

**Prompt:** Create training and validation data loaders, train the model, and calculate the total number of model parameters.

In [7]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=256)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Total model parameters: {parameter_count:,}")
history = train2(model, optimizer, xentropy, metric, train_loader,
                 validation_loader, n_epochs=5, device=device)


Total model parameters: 203,530
Epoch 1/5, train loss: 0.5237, train accuracy: 0.8152, validation accuracy: 0.8524
Epoch 2/5, train loss: 0.3828, train accuracy: 0.8613, validation accuracy: 0.8558
Epoch 3/5, train loss: 0.3419, train accuracy: 0.8757, validation accuracy: 0.8600
Epoch 4/5, train loss: 0.3141, train accuracy: 0.8857, validation accuracy: 0.8652
Epoch 5/5, train loss: 0.2972, train accuracy: 0.8908, validation accuracy: 0.8798


## 8. Evaluate on FashionMNIST test data

**Prompt:** Load the separate FashionMNIST test dataset, switch the model to evaluation mode through the evaluation function, and compute test accuracy.

In [8]:
full_test_dataset = FashionMNIST(
    root="./data", train=False, download=True, transform=ToTensor()
)
test_loader = DataLoader(full_test_dataset, batch_size=256)
test_accuracy = evaluate_tm(model, test_loader, metric, device).item()
print(f"Test accuracy: {test_accuracy:.4f}")


Test accuracy: 0.8761


## 9. Inspect predictions and softmax probabilities

**Prompt:** For 10 test examples, get the predicted `y` index from the largest logit. Apply softmax and round probabilities to three decimal places. Also display the top four predicted indexes, class names, and probabilities, then check each prediction against its true label. Use the full training dataset’s class names to look up labels.

In [9]:
model.eval()
with torch.no_grad():
    for sample_index in range(min(10, len(full_test_dataset))):
        x_eval, y_eval = full_test_dataset[sample_index]
        logits = model(x_eval.unsqueeze(0).to(device))
        predicted_y = logits.argmax(dim=1).item()
        probabilities = F.softmax(logits, dim=1).squeeze(0)
        rounded_probabilities = [round(value, 3) for value in probabilities.tolist()]
        top_probabilities, top_indexes = torch.topk(probabilities, k=4)
        top_four = [
            (index.item(), full_train_dataset.classes[index.item()],
             round(probability.item(), 3))
            for probability, index in zip(top_probabilities, top_indexes)
        ]
        predicted_label = full_train_dataset.classes[predicted_y]
        actual_label = full_train_dataset.classes[y_eval]
        print(f"Sample {sample_index}: predicted index={predicted_y} "
              f"({predicted_label}), actual index={y_eval} ({actual_label}), "
              f"correct={predicted_y == y_eval}")
        print(f"  Softmax probabilities by class index: {rounded_probabilities}")
        print(f"  Top 4 (index, class, probability): {top_four}")


Sample 0: predicted index=9 (Ankle boot), actual index=9 (Ankle boot), correct=True
  Softmax probabilities by class index: [0.0, 0.0, 0.0, 0.0, 0.0, 0.03, 0.0, 0.023, 0.0, 0.947]
  Top 4 (index, class, probability): [(9, 'Ankle boot', 0.947), (5, 'Sandal', 0.03), (7, 'Sneaker', 0.023), (0, 'T-shirt/top', 0.0)]
Sample 1: predicted index=2 (Pullover), actual index=2 (Pullover), correct=True
  Softmax probabilities by class index: [0.0, 0.0, 0.971, 0.0, 0.006, 0.0, 0.023, 0.0, 0.0, 0.0]
  Top 4 (index, class, probability): [(2, 'Pullover', 0.971), (6, 'Shirt', 0.023), (4, 'Coat', 0.006), (0, 'T-shirt/top', 0.0)]
Sample 2: predicted index=1 (Trouser), actual index=1 (Trouser), correct=True
  Softmax probabilities by class index: [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Top 4 (index, class, probability): [(1, 'Trouser', 1.0), (0, 'T-shirt/top', 0.0), (3, 'Dress', 0.0), (2, 'Pullover', 0.0)]
Sample 3: predicted index=1 (Trouser), actual index=1 (Trouser), correct=True
  Softmax 

## Summary of the dialog from CodeX:

- Loaded FashionMNIST with `torchvision` and split its 60,000 training examples into 55,000 training and 5,000 validation examples.
- Inspected an image tensor’s shape and data type, and mapped its numeric label to a class name.
- Built a 10-class neural network, used cross-entropy loss, and created an Adam optimizer.
- Added training and accuracy tracking, then evaluated on the separate test dataset.
- Printed predicted and actual labels, per-class softmax probabilities, and the top four class predictions with probabilities rounded to three decimal places.
- Counted the model’s parameters.